# Warp Scene Inspection

Interactive, cell-by-cell notebook for inspecting Qianyi_DP simulation scenes with Warp. Run the cells top to bottom; each cell is independent after the setup cell.

> Degradation notice: if Warp discovers no CUDA device on this machine, the final cell degrades to a data-only summary instead of a Warp scene (machine-specific environment details live in `LOCAL_DEV.md`, which is gitignored).

In [5]:
from conftest import _resolve_qydp
# Environment setup: locate and import the Qianyi_DP module.
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for p in (REPO_ROOT, REPO_ROOT / 'tests'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

qydp = _resolve_qydp()

print('Qianyi_DP', getattr(qydp, '__version__', 'dev'), 'loaded; has test:', hasattr(qydp, 'test'))
print('pyd location note: machine-specific paths are recorded in LOCAL_DEV.md (gitignored)')

Qianyi_DP dev loaded; has test: True
pyd location note: machine-specific paths are recorded in LOCAL_DEV.md (gitignored)


In [6]:
# Procedural scene construction (no Blender dependency).
from harness.meshspec import MeshSpec

spec = MeshSpec(rows=10, cols=10, fixed_vertex_indices=(0, 9, 90, 99))
mesh = spec.to_dict()
print('vertices:', len(mesh['vertices']) // 3)
print('edges:', len(mesh['edges']) // 2)
print('triangles:', len(mesh['triangles']) // 3)

vertices: 100
edges: 261
triangles: 162


In [7]:
# Driver collection: Blender-equivalent semantics (PDNewton, 24fps).
from harness.driver import SimDriver

driver = SimDriver(qydp, fps=24, frames=30)
run = driver.run(spec.to_input_data())
print('local frames:', run.local_frames.shape)
print('world frames:', run.world_frames.shape)

local frames: (30, 100, 3)
world frames: (30, 100, 3)


In [8]:
# Frame data checks: finiteness and per-frame statistics.
from harness.traces import compute_frame_stats

stats = compute_frame_stats(run.local_frames, run.world_frames, pinned_indices=spec.fixed_vertex_indices)
print('all finite:', all(s['all_finite'] for s in stats))
print('last frame stats:', stats[-1])

all finite: True
last frame stats: {'frame': 29, 'max_disp': 0.44623127579689026, 'mean_disp': 0.20015668869018555, 'non_finite_count': 0, 'all_finite': True, 'max_pinned_drift': 0.0, 'z_min': 0.0, 'z_max': 0.015282505191862583, 'world_max_abs': 1.0}


In [9]:
# Interactive Warp scene (degrades gracefully when no CUDA device is available).
import numpy as np

try:
    import warp as wp
    wp.init()
    devices = [d for d in wp.get_devices() if 'cuda' in str(d).lower()]
except Exception as exc:
    devices = []
    print('Warp unavailable:', exc)

if devices:
    device = devices[0]
    print('Warp CUDA device:', device)
    pts = wp.array(run.local_frames[-1].astype(np.float32), dtype=wp.vec3, device=device)
    print('warp array device:', pts.device, 'shape:', pts.shape)
    # Extend here with an interactive camera/state inspection scene.
else:
    print('Degradation notice: no Warp CUDA device found (see LOCAL_DEV.md);')
    print('skipping the interactive scene - frame data inspection above is the fallback.')

Warp 1.14.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD"
     "cuda:0"   : "NVIDIA GeForce RTX 3070 Laptop GPU" (8 GiB, sm_86, mempool enabled)
   Kernel cache:
     \\?\C:\Users\26jjk\AppData\Local\NVIDIA\warp\Cache\1.14.0
Warp CUDA device: cuda:0
warp array device: cuda:0 shape: (100,)
